In [2]:
!pip install gin-config polars einops tqdm -q
!git clone -b global_temp_split https://github.com/mikhaildanilov/deep-dive-into-Recommender-Systems-with-Generative-Retrieval-article.git
%cd deep-dive-into-Recommender-Systems-with-Generative-Retrieval-article

Cloning into 'deep-dive-into-Recommender-Systems-with-Generative-Retrieval-article'...
remote: Enumerating objects: 219, done.
remote: Total 219 (delta 0), reused 0 (delta 0), pack-reused 219 (from 1)
Receiving objects: 100% (219/219), 60.81 MiB | 36.26 MiB/s, done.
Resolving deltas: 100% (105/105), done.
/kaggle/working/deep-dive-into-Recommender-Systems-with-Generative-Retrieval-article


In [3]:
!pip install gdown -q
!gdown 1qGxgmx7G_WB7JE4Cn_bEcZ_o_NAJLE3G -O P5_data.zip

Downloading...
From (original): https://drive.google.com/uc?id=1qGxgmx7G_WB7JE4Cn_bEcZ_o_NAJLE3G
From (redirected): https://drive.google.com/uc?id=1qGxgmx7G_WB7JE4Cn_bEcZ_o_NAJLE3G&confirm=t&uuid=10295af2-1f6d-4cfc-8244-fe2c3679b32c
To: /kaggle/working/deep-dive-into-Recommender-Systems-with-Generative-Retrieval-article/P5_data.zip
100%|███████████████████████████████████████| 1.26G/1.26G [00:10<00:00, 116MB/s]


In [4]:
!unzip -q P5_data.zip
!mkdir -p dataset/amazon/raw
!mv data/* dataset/amazon/raw/
!ls dataset/amazon/raw/

amazon.py  __init__.py	ml32m.py	  processed.py	sports	utils.py
beauty	   ml1m.py	preprocessing.py  schemas.py	toys	yelp


In [5]:
!rm -r dataset/amazon/raw/beauty
!cp -r /kaggle/input/datasets/vyascheslavroshin/temporal-amazon/beauty dataset/amazon/raw/beauty
!ls dataset/amazon/raw/

amazon.py  __init__.py	ml32m.py	  processed.py	sports	utils.py
beauty	   ml1m.py	preprocessing.py  schemas.py	toys	yelp


In [6]:
!ls dataset/amazon/raw/beauty

datamaps.json	      negative_samples.txt	   sequential_data.txt
exp_splits.pkl	      rating_splits_augmented.pkl  user_id2name.pkl
item2attributes.json  review_splits.pkl


In [7]:
import sys; sys.path.insert(0, '.')

In [10]:
# Accelerator: None

!python -m baselines.ease --split beauty

[EASE] split=beauty users=22363 items=12101
[EASE] reg=250.0
[EASE] VAL : recall@5=0.0252, ndcg@5=0.0180, coverage@5=0.4191, recall@10=0.0375, ndcg@10=0.0219, coverage@10=0.6316, recall@50=0.0750, ndcg@50=0.0300, coverage@50=0.9668, recall@100=0.1069, ndcg@100=0.0352, coverage@100=0.9782
[EASE] TEST: recall@5=0.0396, ndcg@5=0.0340, coverage@5=0.4230, recall@10=0.0475, ndcg@10=0.0365, coverage@10=0.6278, recall@50=0.0755, ndcg@50=0.0427, coverage@50=0.9654, recall@100=0.0915, ndcg@100=0.0452, coverage@100=0.9783


In [11]:
# Accelerator: None

print("seed 42:")
!python -m baselines.mf_bpr --split beauty --epochs 50 --seed 42
print("seed 228:")
!python -m baselines.mf_bpr --split beauty --epochs 50 --seed 228
print("seed 1337:")
!python -m baselines.mf_bpr --split beauty --epochs 50 --seed 1337

seed 42:
[MF-BPR] split=beauty users=22363 items=12101 seed=42
[MF-BPR] epoch   5/50 loss=0.0567 val: recall@5=0.0104, ndcg@5=0.0068, coverage@5=0.1729, recall@10=0.0165, ndcg@10=0.0088, coverage@10=0.2524, recall@50=0.0463, ndcg@50=0.0151, coverage@50=0.5813, recall@100=0.0714, ndcg@100=0.0192, coverage@100=0.7665
[MF-BPR] epoch  10/50 loss=0.0165 val: recall@5=0.0102, ndcg@5=0.0067, coverage@5=0.2409, recall@10=0.0163, ndcg@10=0.0086, coverage@10=0.3531, recall@50=0.0464, ndcg@50=0.0151, coverage@50=0.7274, recall@100=0.0720, ndcg@100=0.0192, coverage@100=0.8731
[MF-BPR] epoch  15/50 loss=0.0107 val: recall@5=0.0103, ndcg@5=0.0065, coverage@5=0.2739, recall@10=0.0160, ndcg@10=0.0084, coverage@10=0.3947, recall@50=0.0458, ndcg@50=0.0148, coverage@50=0.7746, recall@100=0.0727, ndcg@100=0.0191, coverage@100=0.8975
[MF-BPR] epoch  20/50 loss=0.0085 val: recall@5=0.0099, ndcg@5=0.0062, coverage@5=0.2983, recall@10=0.0167, ndcg@10=0.0084, coverage@10=0.4207, recall@50=0.0455, ndcg@50=0.014

In [14]:
# Accelerator: GPU T4 x2

print("seed 42:")
!python -m baselines.sasrec --split beauty --epochs 50 --device cuda --seed 42
print("seed 228:")
!python -m baselines.sasrec --split beauty --epochs 50 --device cuda --seed 228
print("seed 1337:")
!python -m baselines.sasrec --split beauty --epochs 50 --device cuda --seed 1337

seed 42:
[SASRec] split=beauty users=22363 items=12101 seed=42
100%|█████████████████████████████████████████| 159/159 [00:04<00:00, 37.66it/s]
[SASRec] epoch   5/50 loss=7.1190 val: recall@5=0.0537, ndcg@5=0.0376, coverage@5=0.3094, recall@10=0.0792, ndcg@10=0.0458, coverage@10=0.4126, recall@50=0.1746, ndcg@50=0.0664, coverage@50=0.7159, recall@100=0.2405, ndcg@100=0.0770, coverage@100=0.8389
100%|█████████████████████████████████████████| 159/159 [00:04<00:00, 33.06it/s]
[SASRec] epoch  10/50 loss=6.4652 val: recall@5=0.0606, ndcg@5=0.0430, coverage@5=0.5566, recall@10=0.0857, ndcg@10=0.0511, coverage@10=0.6994, recall@50=0.1844, ndcg@50=0.0725, coverage@50=0.9311, recall@100=0.2456, ndcg@100=0.0824, coverage@100=0.9672
100%|█████████████████████████████████████████| 159/159 [00:04<00:00, 36.18it/s]
[SASRec] epoch  15/50 loss=6.1699 val: recall@5=0.0644, ndcg@5=0.0460, coverage@5=0.6384, recall@10=0.0893, ndcg@10=0.0540, coverage@10=0.7824, recall@50=0.1822, ndcg@50=0.0741, coverage

In [15]:
# Accelerator: GPU T4 x2

print("seed 42:")
!python -m baselines.bert4rec --split beauty --epochs 50 --device cuda --seed 42
print("seed 228:")
!python -m baselines.bert4rec --split beauty --epochs 50 --device cuda --seed 228
print("seed 1337:")
!python -m baselines.bert4rec --split beauty --epochs 50 --device cuda --seed 1337

seed 42:
[BERT4Rec] split=beauty users=22363 items=12101 seed=42
100%|█████████████████████████████████████████| 175/175 [00:05<00:00, 32.43it/s]
[BERT4Rec] epoch   5/50 loss=8.2092 val: recall@5=0.0239, ndcg@5=0.0151, coverage@5=0.0190, recall@10=0.0378, ndcg@10=0.0195, coverage@10=0.0276, recall@50=0.1122, ndcg@50=0.0354, coverage@50=0.0839, recall@100=0.1687, ndcg@100=0.0445, coverage@100=0.1321
100%|█████████████████████████████████████████| 175/175 [00:05<00:00, 32.99it/s]
[BERT4Rec] epoch  10/50 loss=7.8754 val: recall@5=0.0292, ndcg@5=0.0182, coverage@5=0.0541, recall@10=0.0493, ndcg@10=0.0245, coverage@10=0.0777, recall@50=0.1350, ndcg@50=0.0428, coverage@50=0.1911, recall@100=0.1989, ndcg@100=0.0532, coverage@100=0.2782
100%|█████████████████████████████████████████| 175/175 [00:05<00:00, 33.42it/s]
[BERT4Rec] epoch  15/50 loss=7.7043 val: recall@5=0.0297, ndcg@5=0.0191, coverage@5=0.0914, recall@10=0.0533, ndcg@10=0.0267, coverage@10=0.1337, recall@50=0.1481, ndcg@50=0.0472, 